# Getting started: check your setup (Python)

**Goal.** Confirm that this notebook runs in the Trees from Space environment, that it can write files, whether this machine can reach NASA's data servers, and whether your Earthdata credentials work. The notebook saves a short report of what it found.

**Time.** About 5 minutes. **Data downloaded.** None (one small metadata request when online).

**Before starting.** Create the environment and register the kernel ([environment guide](../../docs/getting-started/environments.md)), and store your Earthdata credentials in `~/.netrc` ([Earthdata Login guide](../../docs/getting-started/earthdata-login.md)). On CHPC, follow the [CHPC guide](../../docs/getting-started/chpc.md). If you are new to notebooks, read [Notebook basics](../../docs/getting-started/notebook-basics.md).

Run the cells in order with **Shift + Enter**. Each check prints `OK` or explains what to fix.

## Settings

Everything you might change is in the next cell. You can leave these values as they are.

In [ ]:
# Check that NASA accepts your Earthdata credentials (needs internet access).
CHECK_EARTHDATA_LOGIN = True

# Open one small NASA data file to test the full download route (needs internet access).
CHECK_DATA_ACCESS = True

# Seconds to wait when testing whether this machine can reach NASA.
NETWORK_TIMEOUT_S = 5

## 1. The environment

The next cell loads the packages the lessons use and prints their versions. In Google Colab, it first installs the packages Colab does not already have, which takes a minute or two.

In [ ]:
import importlib
import json
import netrc
import os
import platform
import socket
import stat
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    colab_packages = ["earthaccess==0.19.0", "pystac-client==0.9.1", "rasterio==1.5.0"]
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *colab_packages], check=True)

REQUIRED = ["earthaccess", "pystac_client", "rasterio", "pyproj", "geopandas", "numpy", "pandas", "matplotlib"]
versions, missing = {}, []
for name in REQUIRED:
    try:
        versions[name] = importlib.import_module(name).__version__
    except ImportError:
        missing.append(name)

print(f"Python {platform.python_version()} at {sys.executable}")
for name, version in versions.items():
    print(f"  {name:<12} {version}")
if missing:
    print(f"\nMissing packages: {', '.join(missing)}.")
    print("Choose the 'Python 3.12 (Trees from Space)' kernel, or create the environment first.")
else:
    import rasterio

    versions["gdal"] = rasterio.__gdal_version__
    print(f"  {'gdal':<12} {versions['gdal']}")
    print("\nOK: all lesson packages are available.")

## 2. Where this notebook runs and saves files

Lessons save downloaded data to a `cache/` folder and results to an `outputs/` folder at the top of the repository. You can move them by setting the `TFS_CACHE_DIR` and `TFS_OUTPUT_DIR` environment variables ([environment guide](../../docs/getting-started/environments.md)). The next cell finds these folders and checks that it can write to them.

In [ ]:
def find_repo_root(start: Path) -> Path | None:
    """Walk up from `start` to the folder holding the .tfs-root marker file."""
    for folder in [start, *start.parents]:
        if (folder / ".tfs-root").exists():
            return folder
    return None


def detect_platform() -> str:
    """Name the kind of machine running this notebook (no hostnames are saved)."""
    if IN_COLAB:
        return "colab"
    host = socket.gethostname()
    on_chpc = "chpc.utah.edu" in socket.getfqdn() or Path("/scratch/general/vast").is_dir()
    if not on_chpc:
        return "local"
    if host.startswith("frisco"):
        return "chpc-frisco"
    partition = os.environ.get("SLURM_JOB_PARTITION")
    return f"chpc-{partition}" if partition else "chpc-login"


PLATFORM = detect_platform()
REPO_ROOT = find_repo_root(Path.cwd())
BASE_DIR = Path("/content/trees-from-space") if IN_COLAB else (REPO_ROOT or Path.cwd())
CACHE_DIR = Path(os.environ.get("TFS_CACHE_DIR", BASE_DIR / "cache"))
OUTPUT_DIR = Path(os.environ.get("TFS_OUTPUT_DIR", BASE_DIR / "outputs"))

print(f"Platform: {PLATFORM}")
if REPO_ROOT is None and not IN_COLAB:
    print("Note: this notebook is not inside the trees-from-space folder; using the current folder.")

paths_writable = True
for folder in (CACHE_DIR, OUTPUT_DIR):
    try:
        folder.mkdir(parents=True, exist_ok=True)
        probe = folder / ".write-test"
        probe.write_text("ok")
        probe.unlink()
        print(f"OK: can write to {folder}")
    except OSError as error:
        paths_writable = False
        print(f"Cannot write to {folder}: {error}")

## 3. Can this machine reach NASA?

NASA data live on servers on the internet. Not every machine can reach them: some CHPC compute nodes have no internet access, although OnDemand sessions on the default interactive partition could reach NASA in our tests. The next cell tries to connect to NASA's search service and stops waiting after a few seconds.

In [ ]:
def network_available(host: str = "cmr.earthdata.nasa.gov", timeout_s: float = 5) -> bool:
    """Return True if a connection to `host` on the HTTPS port succeeds."""
    if os.environ.get("TFS_OFFLINE") == "1":  # lets you test the offline route
        return False
    try:
        with socket.create_connection((host, 443), timeout=timeout_s):
            return True
    except OSError:
        return False


ONLINE = network_available(timeout_s=NETWORK_TIMEOUT_S)
if ONLINE:
    print("OK: this machine can reach NASA. Lessons can search for and download data here.")
elif PLATFORM.startswith("chpc"):
    print("This CHPC machine cannot reach the internet. Some compute nodes have no internet access.")
    print("Lessons that download data need a first run on a machine with internet access:")
    print("see 'Running a lesson that downloads data' in docs/getting-started/chpc.md.")
else:
    print("No connection to NASA. Check your internet connection, then rerun this cell.")

## 4. Earthdata credentials

The lessons read your Earthdata username and password from the file `~/.netrc` in your home directory. The next cell checks that the file exists, that only you can read it, and that it has an entry for NASA Earthdata. It never prints the credentials. In Colab, it offers to create the file for this session.

In [ ]:
NETRC_PATH = Path(os.environ.get("NETRC", Path.home() / ".netrc"))
EARTHDATA_HOST = "urs.earthdata.nasa.gov"


def netrc_status(path: Path) -> dict:
    """Report whether `path` exists, is private, and holds an Earthdata entry."""
    status = {"present": path.exists(), "private": False, "has_earthdata": False}
    if status["present"]:
        status["private"] = stat.S_IMODE(path.stat().st_mode) & 0o077 == 0
        try:
            status["has_earthdata"] = netrc.netrc(path).authenticators(EARTHDATA_HOST) is not None
        except (netrc.NetrcParseError, OSError):
            pass
    return status


if IN_COLAB and not netrc_status(NETRC_PATH)["has_earthdata"] and ONLINE:
    import earthaccess

    try:  # Colab Secrets panel, if you added EARTHDATA_USERNAME and EARTHDATA_PASSWORD
        from google.colab import userdata

        os.environ["EARTHDATA_USERNAME"] = userdata.get("EARTHDATA_USERNAME")
        os.environ["EARTHDATA_PASSWORD"] = userdata.get("EARTHDATA_PASSWORD")
        earthaccess.login(strategy="environment", persist=True)
    except Exception:
        earthaccess.login(strategy="interactive", persist=True)

NETRC = netrc_status(NETRC_PATH)
if not NETRC["present"]:
    print(f"No credentials file at {NETRC_PATH}. See docs/getting-started/earthdata-login.md.")
elif not NETRC["has_earthdata"]:
    print(f"{NETRC_PATH} has no entry for {EARTHDATA_HOST}. See docs/getting-started/earthdata-login.md.")
else:
    print(f"OK: {NETRC_PATH} has an Earthdata entry.")
if NETRC["present"] and not NETRC["private"]:
    print(f"Other users can read {NETRC_PATH}. Fix this with:  chmod 600 ~/.netrc")

## 5. Log in and open one NASA file

When this machine is online, the next cell asks NASA to check your credentials, then opens the header of one Harmonized Landsat Sentinel-2 (HLS) quality file without downloading the whole file. This is the same route the lessons use to read data, through the GDAL library and your `~/.netrc`.

In [ ]:
TEST_URL = (
    "https://data.lpdaac.earthdatacloud.nasa.gov/lp-prod-protected/HLSS30.020/"
    "HLS.S30.T12SVH.2024167T180919.v2.0/HLS.S30.T12SVH.2024167T180919.v2.0.Fmask.tif"
)
COOKIE_DIR = Path.home() / ".cache" / "trees-from-space"
COOKIE_DIR.mkdir(parents=True, exist_ok=True)
COOKIE_DIR.chmod(0o700)
GDAL_OPTIONS = {
    "GDAL_HTTP_NETRC": "YES",
    "GDAL_HTTP_NETRC_FILE": str(NETRC_PATH),
    "GDAL_HTTP_COOKIEFILE": str(COOKIE_DIR / "earthdata-cookies.txt"),
    "GDAL_HTTP_COOKIEJAR": str(COOKIE_DIR / "earthdata-cookies.txt"),
    "GDAL_DISABLE_READDIR_ON_OPEN": "EMPTY_DIR",
    "CPL_VSIL_CURL_ALLOWED_EXTENSIONS": ".tif",
    "GDAL_HTTP_MAX_RETRY": "3",
    "GDAL_HTTP_RETRY_DELAY": "2",
    "GDAL_HTTP_CONNECTTIMEOUT": "10",
    "GDAL_HTTP_TIMEOUT": "60",
}

login_result, access_result = "skipped (offline)", "skipped (offline)"
if ONLINE and NETRC["has_earthdata"]:
    import earthaccess
    import rasterio

    if CHECK_EARTHDATA_LOGIN:
        try:
            auth = earthaccess.login(strategy="netrc")
            login_result = "ok" if auth.authenticated else "failed"
        except Exception as error:
            login_result = f"failed ({type(error).__name__})"
        print(f"Earthdata login: {login_result}")
    if CHECK_DATA_ACCESS:
        try:
            with rasterio.Env(**GDAL_OPTIONS), rasterio.open(f"/vsicurl/{TEST_URL}") as src:
                print(f"Opened {Path(TEST_URL).name}: {src.width} x {src.height} pixels, {src.crs}")
            access_result = "ok"
        except Exception as error:
            access_result = f"failed ({type(error).__name__})"
            print(f"Could not open the test file: {error}")
        print(f"Data access: {access_result}")
elif ONLINE:
    login_result = access_result = "skipped (no credentials)"
    print("Skipped: add your Earthdata credentials first (section 4).")
else:
    print("Skipped: this machine is offline (section 3).")

## 6. Save the report

The last cell saves what this notebook found to `outputs/getting-started/` and prints it. The report contains no usernames, passwords, hostnames, or personal paths, so you can share it when asking for help.

In [ ]:
report = {
    "notebook": "getting-started-python",
    "run_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "platform": PLATFORM,
    "python": platform.python_version(),
    "packages": versions,
    "missing_packages": missing,
    "paths_writable": paths_writable,
    "network": ONLINE,
    "netrc": NETRC,
    "earthdata_login": login_result,
    "data_access": access_result,
}
report_path = OUTPUT_DIR / "getting-started" / "environment-report-python.json"
report_path.parent.mkdir(parents=True, exist_ok=True)
report_path.write_text(json.dumps(report, indent=1) + "\n")
print(f"Saved {report_path}\n")
print(json.dumps(report, indent=1))

## What next

- **Everything OK and online:** open a lesson from the [tutorial index](../README.md) and run it here.
- **Offline on CHPC:** some compute nodes have no internet access. Lessons that download data then need a short transfer job first, as described in the [CHPC guide](../../docs/getting-started/chpc.md).
- **Something failed:** the message above each result says what to fix. Rerun the notebook afterwards with *Restart Kernel and Run All Cells*.